# Exercise 3.1 Work with SAP HANA Cloud Vector Engine from LangChain

In this exercise you will use **LangChain** — a popular Python framework for building AI applications — to interact with SAP HANA Cloud as a **vector store**.

A *vector store* is a database that can store text alongside its numerical representation (an *embedding vector*), enabling **semantic search**: finding documents by meaning rather than exact keyword matches.

By the end of this exercise you will:
1. Connect to SAP HANA Cloud and create a simple vector store from scratch.
2. Add documents, run semantic searches, and apply metadata filters.
3. Expose the `OBJECT_INDEX` table (built in exercise 1) as a LangChain vector store.
4. Use a **Self-Query Retriever** to let an LLM automatically translate natural-language questions into structured database filters.

## Connect to SAP HANA Cloud

First, we establish a database connection using the `hdbcli` Python driver for SAP HANA.

The connection details (host, port, password) are loaded from a `.env` file via `python-dotenv`, so sensitive credentials are not hard-coded in the notebook. Replace `000` in the user name with your assigned student number.

In [1]:
import os
from dotenv import load_dotenv
from hdbcli import dbapi

load_dotenv()

connection = dbapi.connect(
    user="DA263_000", # (replace 000 with your student number, e.g. 001)
    password=os.environ.get("HANA_DB_PASSWORD"),
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    # sslValidateCertificate = False
)

print('HANA Cloud connected:', connection.isconnected())

HANA Cloud connected: True


## Create a Vector Store

In LangChain, a **vector store** is an abstraction over any database that can store and search embedding vectors. For SAP HANA Cloud, this is backed by a regular HANA table with a specific three-column layout:

| Column | Purpose |
|---|---|
| `VEC_TEXT` | The original text content |
| `VEC_META` | JSON metadata attached to the document |
| `VEC_VECTOR` | The embedding vector (the numerical representation of the text) |

Vector stores are the foundation of **Retrieval-Augmented Generation (RAG)**: instead of relying solely on an LLM's training data, you first retrieve relevant documents from the vector store and pass them as context to the LLM — keeping answers grounded in your own data.

The code below creates a `HanaDB` instance backed by the table `LC_HANA_VECTOR_STORE`. Text embeddings are computed automatically inside HANA Cloud using the built-in model `SAP_GXY.20250407`, so no external embedding service is needed.

In [2]:
from langchain_hana import HanaDB
from langchain_hana import HanaInternalEmbeddings

embeddings = HanaInternalEmbeddings(internal_embedding_model_id="SAP_GXY.20250407")

# creating a vector store, i.e. a HanaDB instance, creates a simple table in HANA Cloud with 3 columns: VEC_TEXT, VEC_META, VEC_VECTOR
# If the table already exists, it is checked and used.
vector_store = HanaDB(
    connection=connection, 
    embedding=embeddings, 
    table_name="LC_HANA_VECTOR_STORE"
)

Now we add some sample documents to the vector store. In LangChain, a **`Document`** is a simple container with two fields:
- `page_content` — the text that will be embedded and searched.
- `metadata` — an arbitrary dictionary of key/value pairs you can later use for filtering.

Calling `add_documents()` sends the texts to HANA Cloud, where the embedding model converts them into vectors and stores everything in the `LC_HANA_VECTOR_STORE` table.

In [3]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.",
        metadata={"source": "SAP", "confidence": 1.0},
    ),
    Document(
        page_content="SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.",
        metadata={"source": "SAP", "confidence": 1.0},
    ),
    Document(
            page_content="Sapperlot is a traditional German and regional interjection used to express surprise, amazement, annoyance, or mild indignation.",
            metadata={"source": "Internet", "confidence": 0.8},
        ),
]

vector_store.add_documents(docs)

[]

With 3 documents stored, let's run a **semantic search**. Unlike a keyword search (which looks for exact word matches), a semantic search embeds the query text into a vector and finds the documents whose vectors are closest — meaning documents that are *conceptually similar*, even if they use different words.

`similarity_search_with_score` returns each matching document together with a **cosine similarity score** between 0 and 1. A score closer to 1 means the document is more similar to the query. Notice how *"Sapperlot"* scores lowest — it shares no conceptual overlap with a developer conference.

In [4]:
# scores are cosine similarity in [0, 1]; higher = more similar
docs = vector_store.similarity_search_with_score(
    query="conference for developers", 
    k=3
    )

# print the results
for doc, score in docs:
    print("-" * 80)
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

--------------------------------------------------------------------------------
Score: 0.635400259230992
Content: SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.
Metadata: {'source': 'SAP', 'confidence': 1.0}
--------------------------------------------------------------------------------
Score: 0.5555538488141406
Content: SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.
Metadata: {'source': 'SAP', 'confidence': 1.0}
--------------------------------------------------------------------------------
Score: 0.3332759543080325
Content: Sapperlot is a traditional German and regional interjection used to express surprise, amazement, annoyance, or mild indignation.
Metadata: {'source': 'Internet', 'confidence': 0.8}


Semantic similarity alone is not always enough — you may also want to restrict results based on structured attributes. The vector store supports **metadata filters** using a MongoDB-style syntax (e.g. `$gt`, `$eq`, `$in`). These filters are translated into SQL `WHERE` clauses before the vector search runs, so only matching rows are considered.

In the example below we filter on `confidence > 0.9`, which excludes the *"Sapperlot"* document (confidence 0.8) from the results entirely.

In [5]:
# Define a filter expression on the metadata, e.g. "confidence" > 0.9
# This removes the last document from the result, having a confidence of 0.8
docs = vector_store.similarity_search_with_score(
    query="conference for developers", 
    k=3, 
    filter={"confidence": {'$gt': 0.9}}
    )

for doc, score in docs:
    print("-" * 80)
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

--------------------------------------------------------------------------------
Score: 0.635400259230992
Content: SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.
Metadata: {'source': 'SAP', 'confidence': 1.0}
--------------------------------------------------------------------------------
Score: 0.5555538488141406
Content: SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.
Metadata: {'source': 'SAP', 'confidence': 1.0}


So far we called the vector store directly. LangChain also defines a **Retriever** interface — a standardised wrapper that exposes any data source (vector store, search engine, database, etc.) through a single `invoke(query)` method.

Wrapping the vector store as a retriever is important because LangChain's built-in chains and agents expect this interface. Once wrapped, the retriever can be plugged into a RAG pipeline without any further changes.

In [6]:
retriever = vector_store.as_retriever()

docs = retriever.invoke(
    input="developer conference", 
    filter={"confidence": {'$gt': 0.9}}
    )
for doc in docs:
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

Score: 0.5555538488141406
Content: SAP TechEd is SAP's premier annual technical conference designed for developers, engineers, architects, and IT professionals.
Metadata: {'source': 'SAP', 'confidence': 1.0}
Score: 0.5555538488141406
Content: SAP Sapphire is the flagship annual business technology conference hosted by SAP, one of the world's largest enterprise software companies.
Metadata: {'source': 'SAP', 'confidence': 1.0}


## Expose an Existing HANA Cloud Table as Vector Store

In exercise 1 we pre-populated the `OBJECT_INDEX` table with descriptions of every database object (tables, views, functions) in the `ERGAST_2025` schema — each row already contains a pre-computed embedding vector.

To use this existing table as a LangChain vector store, we pass the column names explicitly so `HanaDB` knows where to find the text, metadata, and vector.

The `specific_metadata_columns` parameter is worth noting: it tells `HanaDB` to treat certain columns (like `SCHEMA_NAME` or `OBJECT_TYPE`) as **first-class filter columns** rather than packing them into the generic JSON `LC_METADATA` column. This means filter predicates such as `SCHEMA_NAME = 'ERGAST_2025'` are evaluated directly by the HANA SQL engine — much faster than parsing JSON for every row.

In [7]:
from langchain_hana import HanaDB
from langchain_hana import HanaInternalEmbeddings

embeddings = HanaInternalEmbeddings(internal_embedding_model_id="SAP_GXY.20250407")

vector_store = HanaDB(
    connection=connection,
    embedding=embeddings,
    table_name="OBJECT_INDEX",
    content_column="DESCRIPTION",
    metadata_column="LC_METADATA",
    vector_column="DESCRIPTION_EMBEDDING",
    specific_metadata_columns=["OBJECT_OID", "SCHEMA_NAME", "OBJECT_NAME", "OBJECT_TYPE"],
)

Now let's try a semantic search against the object index. We search for *"f1 pilots"* — note that the `DRIVERS` table description doesn't contain those exact words. The vector search finds it anyway because the embedding model understands that *"pilots"* and *"drivers"* are conceptually related in a Formula 1 context.

We also apply a metadata filter (`SCHEMA_NAME = 'ERGAST_2025'`) to limit results to that schema only.

In [8]:
query = "f1 pilots"

advanced_filter = {"SCHEMA_NAME": "ERGAST_2025"}

docs = vector_store.similarity_search_with_score(
    query=query, 
    k=3, 
    filter=advanced_filter
    )
for doc, score in docs:
    print("-" * 80)
    print(f"Score: {score}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")

--------------------------------------------------------------------------------
Score: 0.6247333665445782
Metadata: {'OBJECT_OID': 166719, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'DRIVERS', 'OBJECT_TYPE': 'TABLE'}
Content: Contains information about Formula 1 drivers, including their name, date of birth, nationality, driver code, and permanent race number. Each row represents a unique driver.
--------------------------------------------------------------------------------
Score: 0.5888581136077121
Metadata: {'OBJECT_OID': 166740, 'SCHEMA_NAME': 'ERGAST_2025', 'OBJECT_NAME': 'LAPTIMES', 'OBJECT_TYPE': 'TABLE'}
Content: Records the lap time for every driver on every lap of each Formula 1 race. Each row represents a single lap by a driver, capturing the lap number, time, position on track at the end of the lap, and duration in milliseconds.
--------------------------------------------------------------------------------
Score: 0.5826902400551153
Metadata: {'OBJECT_OID': 166753, 'SCH

In the previous step we had to manually specify the filter `SCHEMA_NAME = 'ERGAST_2025'`. In a real application, users will type plain English — not structured filter expressions. This is where the **Self-Query Retriever** comes in.

A Self-Query Retriever passes the user's natural-language query to an LLM, which analyses it and automatically extracts:
1. The **semantic search part** (what to look for in the embeddings).
2. The **filter part** (any structured conditions, e.g. *"only views"* → `OBJECT_TYPE = 'VIEW'`).

The LLM we use here is **GPT-4.1**, running on SAP AI Core and accessed through the `gen_ai_hub` SDK, which integrates it as a standard LangChain chat model.

In [9]:
from gen_ai_hub.proxy.langchain import ChatOpenAI
from gen_ai_hub.proxy import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

llm = ChatOpenAI(proxy_model_name='gpt-4.1', proxy_client=proxy_client)

print("LLM model:", llm.model_name)

LLM model: gpt-4.1


For the LLM to generate correct filters, it needs to know what columns are available and what they contain. We describe each filterable column using `AttributeInfo` objects — essentially a schema hint passed to the LLM in its system prompt.

We also provide a `document_content_description` that tells the LLM what the text content represents. The `HanaTranslator` is a HANA-specific component that converts the LLM's structured query output into the filter syntax understood by `HanaDB`.

In [10]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_hana import HanaTranslator

metadata_field_info = [
    AttributeInfo(
        name="OBJECT_OID",
        description="The ID of the database object.",
        type="integer",
    ),
    AttributeInfo(
        name="SCHEMA_NAME",
        description="The schema name where the object resides.",
        type="string",
    ),
    AttributeInfo(
        name="OBJECT_NAME",
        description="The name of the database object.",
        type="string",
    ),
    AttributeInfo(
        name="OBJECT_TYPE",
        description="The type of the database object, e.g. TABLE, VIEW, FUNCTION.",
        type="string",
    ),
]

document_content_description = "A catalog of database objects."

hana_translator = HanaTranslator()

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=hana_translator,
)

Now we invoke the Self-Query Retriever with a plain English question. The LLM parses *"which views are in the system with data about results or races"* and automatically derives:
- **Filter:** `OBJECT_TYPE = 'VIEW'` (from the word *"views"*)
- **Semantic query:** `"results or races"` (the topical part of the question)

No manual filter construction required — the LLM does it for us.

In [11]:
query_prompt = "which views are in the system with data about results or races"
# query_prompt = "tables in the ERGAST_2025 schema f1 pilots"

docs = retriever.invoke(
    input=query_prompt
    )

for doc in docs:
    print("-" * 80)
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")

--------------------------------------------------------------------------------
Metadata: {'OBJECT_OID': 167150, 'SCHEMA_NAME': 'DA263_000', 'OBJECT_NAME': 'V_RESULTS', 'OBJECT_TYPE': 'VIEW'}
Content: # Object Description

**Formula 1 Race Results with Driver and Constructor Details**

This is a denormalized view that combines race results with associated driver, constructor, race, and status information from the ERGAST 2025 Formula 1 dataset. 

**Key Content:**
- Race performance metrics (grid position, finishing position, points awarded)
- Driver information (full name)
- Race details (name, date)
- Constructor/team information
- Race completion status

**Primary Use Cases:**
- Analyzing driver performance across races
- Comparing constructor/team standings and results
- Tracking historical race outcomes and results
- Studying driver career statistics and race-by-race performance

**Note:** The view structure appears to be a SELECT query rather than a stored view definition, showing

Curious what the LLM actually produced? This cell exposes the intermediate steps by running the query construction chain directly, without executing the search.

The output shows:
- The **structured query** as parsed by the LLM (semantic part + filter expression).
- The **translated filter** in the `HanaDB` MongoDB-style syntax that gets sent to the vector store.

This is useful for debugging — if the retriever returns unexpected results, you can inspect the generated filter here to see whether the LLM interpreted your question correctly.

In [12]:
from langchain_classic.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

sq = query_constructor.invoke(input=query_prompt)

print("Structured query:\n", sq)

print("\nTranslated for HANA vector store:\n", hana_translator.visit_structured_query(sq))

Structured query:
 query='results or races' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='OBJECT_TYPE', value='VIEW') limit=None

Translated for HANA vector store:
 ('results or races', {'filter': {'OBJECT_TYPE': {'$eq': 'VIEW'}}})
